# S36 Qwen3-Reranker-4B on Kaggle (2x T4): final-4b

Scores the S36 subset `final-4b` (47,833 pairs, about 2.3 h on 2x T4): the unscored pairs of the cosine top-300 of every query (free frame) and of the pool13 top-n_q (EF2-cut-p4b frame), plus the new 120ask pairs of `pairs_new_final_v2` (b0008, b0009).
Same scorer (`judge_score.py`, SHA `688948a7...`), same model commit (`22e68366...`), same fp16 / hf settings as S31 and S37.

**Settings**: Accelerator = GPU T4 x2, Internet = On (the model is public; no HF token or other secret is used).
Run with *Save Version -> Save & Run All (Commit)*.

**Inputs** (Add Input): ONLY the dataset made from the folder `out/runs/s36-fuse4/d300/kaggle-dataset-final4b/` (`judge_score.py` + `final-4b/`). Do not attach any older S31 / S37 dataset: two judge_score.py files stop the notebook.
If a session is interrupted, start a new one and also add the previous session's **output** as an input: finished parts
(`final-4b/qwen3-reranker-4b/parts`) are found automatically and reused; do not add anything else.

Output: `/kaggle/working/final-4b/qwen3-reranker-4b/scores-extra.parquet` and `collect-extra-report.json`.


In [ ]:
# ---- configuration ----
MODEL = "qwen3-reranker-4b"
REVISION = "22e683669bc0f0bd69640a1354a6d0aebcfeede5"   # same 4B commit as S31
DTYPE = "float16"                 # T4 has no native bf16
BACKEND = "hf"
TAGS = ["final-4b"]
EXPECT = {"final-4b": 47833}   # pairs per subset (manifest.json)
WORKERS = 2                       # one process per T4
MAX_BATCH_TOKENS = 16384
RUN_PILOT = False                 # 200-pair speed check on the first subset (does not write parts)
SESSION_SECONDS = 11 * 3600 - 1800   # stop starting parts ~30 min before the 12 h limit (load + collect margin)
OUT_ROOT = "/kaggle/working"

In [ ]:
import glob, hashlib, json, os, subprocess, sys, time
from pathlib import Path
T0 = time.time()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch, transformers, pandas, pyarrow
print("torch", torch.__version__, "transformers", transformers.__version__, "pandas", pandas.__version__, "pyarrow", pyarrow.__version__, "gpus", torch.cuda.device_count())
assert torch.cuda.device_count() >= WORKERS, "turn on GPU T4 x2"
from packaging.version import Version
if Version(transformers.__version__) < Version("4.51.0"):   # Qwen3 support
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.51.0"], check=True)
    print("upgraded transformers; restart the kernel and run again")

scripts = glob.glob("/kaggle/input/**/judge_score.py", recursive=True)
assert len(scripts) == 1, f"add the dataset made from kaggle-dataset-final4b (one judge_score.py): {scripts}"
SCRIPT = scripts[0]
DATA = str(Path(SCRIPT).parent)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()

assert sha256(SCRIPT) == "688948a7c93ac01ab7c18c565755318ceec5cb041c6663ec9204f7f5695a7186", "judge_score.py is not the S31 pool-extra scorer"
POOL = {}
for tag in TAGS:
    d = f"{DATA}/{tag}"
    man = json.load(open(f"{d}/manifest.json"))
    assert man["pairs"] == EXPECT[tag] and man["revision"] == REVISION, (tag, man["pairs"], man["revision"])
    assert sha256(f"{d}/extra_pairs.parquet") == man["extra_pairs_sha256"], f"{tag}: extra_pairs.parquet SHA mismatch"
    shard = glob.glob(f"{d}/shards/pool-*.parquet")
    assert len(shard) == 1 and sha256(shard[0]) == man["shard_sha256"], f"{tag}: shard SHA mismatch"
    POOL[tag] = d
    print(tag, "pairs", man["pairs"], "queries", man.get("queries_needing_4b"), "ETA h (2xT4, 5.8 pairs/s)", man["eta_hours_kaggle_2xT4"])
print("script", SCRIPT)

In [ ]:
# Download once (public model, no token) so the two workers do not race on the cache.
from huggingface_hub import snapshot_download
t0 = time.time()
path = snapshot_download(repo_id="Qwen/Qwen3-Reranker-4B", revision=REVISION)
RESOLVED = Path(path).name
assert RESOLVED == REVISION, f"snapshot commit {RESOLVED} != {REVISION}"
print("Qwen3-Reranker-4B snapshot", path, "commit", RESOLVED, f"{time.time() - t0:.0f}s")

In [ ]:
def resume_roots(tag):
    """Earlier sessions' output added as an input: any .../<tag>/<MODEL>/parts is a resume root (= the dir holding <MODEL>)."""
    roots = []
    for parts_dir in glob.glob(f"/kaggle/input/**/{tag}/{MODEL}/parts", recursive=True):
        roots.append(str(Path(parts_dir).parent.parent))
    revs = set()
    for r in roots:
        for meta in glob.glob(f"{r}/{MODEL}/parts/*.json"):
            revs.add(json.load(open(meta)).get("revision"))
    revs.discard(None)
    assert revs <= {REVISION}, f"earlier parts of {tag} use model commits {revs}, not {REVISION}"
    return roots

def args_for(tag):
    resume = [x for r in resume_roots(tag) for x in ("--resume-from", r)]
    return ["--pool", POOL[tag], "--out", f"{OUT_ROOT}/{tag}", "--pairs-file", f"{POOL[tag]}/extra_pairs.parquet", "--subset", "extra", *resume]

def status(tag):
    res = subprocess.run([sys.executable, SCRIPT, "status", *args_for(tag), "--model", MODEL], capture_output=True, text=True, check=True)
    return json.loads(res.stdout.strip().splitlines()[-1])

for tag in TAGS:
    os.makedirs(f"{OUT_ROOT}/{tag}", exist_ok=True)
    print(tag, "resume roots", resume_roots(tag), "status", status(tag))

In [ ]:
if RUN_PILOT:   # speed only: 200 pairs of the first subset on GPU 0
    tag = TAGS[0]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "TOKENIZERS_PARALLELISM": "false"}
    cmd = [sys.executable, SCRIPT, "pilot", "--pool", POOL[tag], "--out", f"{OUT_ROOT}/pilot", "--model", MODEL, "--backend", BACKEND, "--dtype", DTYPE,
           "--max-batch-tokens", str(MAX_BATCH_TOKENS), "--revision", RESOLVED, "--pairs", "200"]
    subprocess.run(cmd, env=env, check=True)
    pilot = json.load(open(f"{OUT_ROOT}/pilot/{MODEL}/pilot-200-{BACKEND}-{DTYPE}.json"))
    rate = pilot["pairs_per_second"]
    print(json.dumps({k: pilot[k] for k in ("pairs_per_second", "seconds", "prompt_tokens_mean", "peak_cuda_allocated_mib", "gpu") if k in pilot}, indent=2))
    print("ETA both subsets with", WORKERS, "GPUs: %.1f h" % (sum(EXPECT.values()) / (rate * WORKERS) / 3600))

In [ ]:
def run_subset(tag):
    left = SESSION_SECONDS - (time.time() - T0)
    if left < 600:
        print(tag, "not started: session time budget used"); return
    procs = []
    for w in range(WORKERS):
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(w), "TOKENIZERS_PARALLELISM": "false"}
        cmd = [sys.executable, SCRIPT, "score", *args_for(tag), "--model", MODEL, "--backend", BACKEND, "--dtype", DTYPE,
               "--max-batch-tokens", str(MAX_BATCH_TOKENS), "--revision", RESOLVED, "--worker", str(w), "--workers", str(WORKERS),
               "--max-seconds", str(int(left))]
        procs.append(subprocess.Popen(cmd, env=env, stdout=open(f"{OUT_ROOT}/{tag}/worker-{w}.stdout", "w"), stderr=subprocess.STDOUT))
    while any(p.poll() is None for p in procs):
        time.sleep(300)
        st = status(tag)
        print(f"[{tag}] parts {st['parts_done']}/{st['parts_total']}  pairs {st['pairs_done']}/{st['pairs_total']}  elapsed {(time.time() - T0) / 3600:.2f} h", flush=True)
    print(tag, "worker exit codes", [p.returncode for p in procs])
    st = status(tag)
    print(tag, st)
    if st["complete"]:
        subprocess.run([sys.executable, SCRIPT, "collect", *args_for(tag), "--model", MODEL], check=True)
        print(f"[{tag}] done: {OUT_ROOT}/{tag}/{MODEL}/scores-extra.parquet")
    else:
        print(f"[{tag}] INCOMPLETE: save this version, add its output as an input, and run again")

for tag in TAGS:          # strictly one after the other
    run_subset(tag)

In [ ]:
# Final check inside Kaggle: per subset, pairs scored == pairs expected, same pair_id set, no NaN, SHA256 pinned in the report.
for tag in TAGS:
    p = f"{OUT_ROOT}/{tag}/{MODEL}/scores-extra.parquet"
    if not os.path.exists(p):
        print(tag, "no scores-extra.parquet yet"); continue
    s = pandas.read_parquet(p)
    ex = pandas.read_parquet(f"{POOL[tag]}/extra_pairs.parquet")
    rep = json.load(open(f"{OUT_ROOT}/{tag}/{MODEL}/collect-extra-report.json"))
    print(tag, "rows", len(s), "expected", EXPECT[tag], "unique pair_id", s.pair_id.nunique(), "same ids", set(s.pair_id) == set(ex.pair_id),
          "NaN", int(s.score.isna().sum()), "score min/max", float(s.score.min()), float(s.score.max()))
    print("  sha256", sha256(p), "report", rep["scores_sha256"], "revisions", rep["revisions"], "dtypes", rep["dtypes"], "gpu s", rep["scoring_gpu_seconds_total_all_parts"])
    assert len(s) == EXPECT[tag] and set(s.pair_id) == set(ex.pair_id) and sha256(p) == rep["scores_sha256"]
print("Download /kaggle/working/final-4b/qwen3-reranker-4b/scores-extra.parquet and collect-extra-report.json")